In [2]:
!pip install numpy

In [5]:
from __future__ import print_function
import numpy as np
import pandas as pd
import tensorflow as tf
from keras import layers
from keras import regularizers
from keras.models import Model
from keras.models import Sequential
from keras.layers import *
from keras.regularizers import l1,l2, L1L2
import keras
from keras.optimizers import SGD
from keras.callbacks import EarlyStopping, Callback, ModelCheckpoint, ReduceLROnPlateau
from scipy.stats import pearsonr

RuntimeError: module compiled against API version 0x10 but this version of numpy is 0xf

SystemError: initialization of _pywrap_checkpoint_reader raised unreported exception

In [2]:
##one of K encoding

nb_classes = 4

def indices_to_one_hot(data,nb_classes):

	targets = np.array(data).reshape(-1)

	return np.eye(nb_classes)[targets]

In [3]:
def readData(input):
    # Read the data
    data = pd.read_csv(input, sep='\t', header=0, na_values='nan')

    # Convert SNP, pheno, and folds columns to numeric, coercing errors to NaN
    SNP = data.iloc[:, 4:].apply(pd.to_numeric, errors='coerce').values
    pheno = pd.to_numeric(data.iloc[:, 1], errors='coerce').values
    folds = pd.to_numeric(data.iloc[:, 0], errors='coerce').values

    # Initialize the array to store one-hot encoded SNP data
    arr = np.empty(shape=(SNP.shape[0], SNP.shape[1], nb_classes))

    # Iterate over the rows to convert SNP values to one-hot encoding
    for i in range(0, SNP.shape[0]):
        arr[i] = indices_to_one_hot(pd.to_numeric(SNP[i], downcast='signed'), nb_classes)

    return arr, pheno, folds

In [4]:
def resnet(input):

	inputs = Input(shape=(input.shape[1],nb_classes))


	x = Conv1D(10,4,padding='same',activation = 'linear',kernel_initializer = 'TruncatedNormal', kernel_regularizer=regularizers.l2(0.1),bias_regularizer = regularizers.l2(0.01))(inputs)

	x = Conv1D(10,20,padding='same',activation = 'linear', kernel_initializer = 'TruncatedNormal',kernel_regularizer=regularizers.l2(0.1),bias_regularizer = regularizers.l2(0.01))(x)

	x = Dropout(0.75)(x)

	shortcut = Conv1D(10,4,padding='same',activation = 'linear',kernel_initializer = 'TruncatedNormal', kernel_regularizer=regularizers.l2(0.1),bias_regularizer = regularizers.l2(0.01))(inputs)
	x = layers.add([shortcut,x])

	x = Conv1D(10,4,padding='same',activation = 'linear',kernel_initializer = 'TruncatedNormal', kernel_regularizer=regularizers.l2(0.1),bias_regularizer = regularizers.l2(0.01))(x)

	x = Dropout(0.75)(x)
	x = Flatten()(x)

	x = Dropout(0.75)(x)

	outputs = Dense(1,activation = isru,bias_regularizer = regularizers.l2(0.01),kernel_initializer = 'TruncatedNormal',name = 'out')(x)

	model = Model(inputs = inputs,outputs = outputs)
	model.compile(loss='mean_squared_error',optimizer=keras.optimizers.Adam(learning_rate=0.001),metrics=['mae'])

	return model

In [5]:
a = 0.02

def isru(x):
    return x / (tf.sqrt(1 + a * tf.square(x)))

In [6]:
def model_train(test, val, train, testPheno, valPheno, trainPheno, model_save, weights_save):

	batch_size = 250
	early_stop = 5
	epoch = 10
	early_stopping = EarlyStopping(monitor='val_mae', mode='min', patience=early_stop, verbose=1, restore_best_weights=True)

	model = resnet(train)
	history = model.fit(train, trainPheno, batch_size=batch_size, epochs=epoch, validation_data=(val, valPheno), callbacks=[early_stopping], shuffle= True)

	#model.save(model_save)
	#model.save_weights(weights_save)

	pred = model.predict(test)
	pred.shape = (pred.shape[0],)
	corr = pearsonr(pred,testPheno)[0]

	return history,corr


In [7]:
def main(IMP_input, QA_input):
    IMP_corr = []
    QA_corr = []

    imp_SNP, imp_pheno, folds = readData(IMP_input)
    QA_SNP, QA_pheno, folds = readData(QA_input)

    PHENOTYPE = imp_pheno

    for i in range(1, 2):

        testIdx = np.where(folds == i)
        if i == 10:
            valIdx = np.where(folds == 1)
            trainIdx = np.intersect1d(np.where(folds != i), np.where(folds != 1))
        else:
            valIdx = np.where(folds == i + 1)
            trainIdx = np.intersect1d(np.where(folds != i), np.where(folds != i + 1))

        trainSNP, trainSNP_QA, trainPheno = imp_SNP[trainIdx], QA_SNP[trainIdx], PHENOTYPE[trainIdx]
        valSNP, valSNP_QA, valPheno = imp_SNP[valIdx], QA_SNP[valIdx], PHENOTYPE[valIdx]
        testSNP, testSNP_QA, testPheno = imp_SNP[testIdx], QA_SNP[testIdx], PHENOTYPE[testIdx]

        history, corr = model_train(testSNP, valSNP, trainSNP, testPheno, valPheno, trainPheno,
                                    'model_IMP/model_' + str(i) + '.txt', 'model_IMP/model_weights' + str(i) + '.h5')
        IMP_corr.append(float('%0.4f' % corr))

        history, corr = model_train(testSNP_QA, valSNP_QA, trainSNP_QA, testPheno, valPheno, trainPheno,
                                    'model_QA/model_' + str(i) + '.txt', 'model_QA/model_weights' + str(i) + '.h5')
        QA_corr.append(float('%0.4f' % corr))

    print("Average PCC (imputed) from 10-fold cross validation: " + str(np.mean(IMP_corr)))
    print("Average PCC (non-imputed) from 10-fold cross validation: " + str(np.mean(QA_corr)))


In [8]:
if __name__ == '__main__':

	#os.chdir("MOISTURE")

	IMP_input =  "IMP_protein.txt"
	QA_input = "QA_protein.txt"

	main(IMP_input,QA_input)


Epoch 1/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 23s 1s/step - loss: 2.5182 - mae: 1.1048 - val_loss: 1.4741 - val_mae: 0.8004
Epoch 2/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 18s 1s/step - loss: 1.6025 - mae: 0.8457 - val_loss: 1.4109 - val_mae: 0.7978
Epoch 3/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 20s 1s/step - loss: 1.4437 - mae: 0.7976 - val_loss: 1.3538 - val_mae: 0.7944
Epoch 4/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 18s 1s/step - loss: 1.3493 - mae: 0.7815 - val_loss: 1.3025 - val_mae: 0.7908
Epoch 5/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 18s 1s/step - loss: 1.3107 - mae: 0.7778 - val_loss: 1.2568 - val_mae: 0.7876
Epoch 6/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 17s 986ms/step - loss: 1.2813 - mae: 0.7845 - val_loss: 1.2157 - val_mae: 0.7834
Epoch 7/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 18s 1s/step - loss: 1.2470 - mae: 0.7777 - val_loss: 1.1767 - val_mae: 0.7811
Epoch 8/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 18s 1s/step - loss: 1.2303 - mae: 0.7851 - val_loss: 1.1431 - val_mae: 0.7777
Epoch 9/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 20s 1s/step - loss: 1.1656 - ma